it extreact the required items first then will do the removal keyowrd search
- it first extract the description, androidmanifest path and readme then perform the search for removal keyword


In [6]:
import os
import re
import requests
import pandas as pd
from base64 import b64decode
from dotenv import load_dotenv
from time import sleep

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")
token_index = 0

def get_auth_header():
    global token_index
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    return {"Authorization": f"token {token}"}

# === Output setup ===
OUTPUT_DIR = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline"
os.makedirs(OUTPUT_DIR, exist_ok=True)

input_path = os.path.join(OUTPUT_DIR, "step2_manifest_final_output.csv")
output_path = os.path.join(OUTPUT_DIR, "step3_removal_keyword_output.csv")

# === Define keyword baskets ===
keywords = ['example', 'sample', 'demo', 'test', 'debug', 'presentation', 'module', 'components', 'lib', 'library',
            'sdk', 'utils', 'utility', 'plugin', 'widget', 'playground', 'framework', 'architecture', 'skeleton',
            'collection', 'starting point', 'protocol', 'benchmark', 'hackathon', 'classroom', 'course', 'exercise',
            'assignment', 'homework', 'assessment', 'interview', 'asset', 'template', 'catalog', 'tutorial', 'tool']
preceded_by = ['this', 'is a', 'is an', 'our', 'my']
not_preceded_by = ['using', 'with']

# === GitHub API functions ===
def fetch_repo_metadata(repo_full_name):
    url = f"https://api.github.com/repos/{repo_full_name}"
    resp = requests.get(url, headers=get_auth_header())
    return resp.json().get("description", "") if resp.status_code == 200 else ""

def fetch_manifest_paths(repo_full_name):
    url = f"https://api.github.com/search/code?q=filename:AndroidManifest.xml+repo:{repo_full_name}"
    resp = requests.get(url, headers=get_auth_header())
    return [item["path"] for item in resp.json().get("items", [])] if resp.status_code == 200 else []

def fetch_readme_content(repo_full_name):
    url = f"https://api.github.com/repos/{repo_full_name}/readme"
    resp = requests.get(url, headers=get_auth_header())
    if resp.status_code == 200:
        content = resp.json().get("content", "")
        return b64decode(content).decode('utf-8', errors='ignore')
    return ""

# === Load data ===
if os.path.exists(output_path):
    df = pd.read_csv(output_path)
else:
    df = pd.read_csv(input_path)
    df["Repository"] = df["html_url"].apply(lambda url: '/'.join(url.strip('/').split('/')[-2:]))
    df["removal_keyword_flag"] = "none"
    df["removal_reason"] = "none"
    df["Valid_Repo_Step3"] = "none"

# === Filter only repos to review ===
to_review = df[(df["Valid_Repo_Step2"] == "yes") & ((df["Valid_Repo_Step3"] == "none") | (df["Valid_Repo_Step3"].isna()))]

for i, idx in enumerate(to_review.index, 1):
    row = df.loc[idx]
    repo = row['Repository']
    print(f"[{i}/{len(to_review)}] Reviewing {repo}...")

    removal_sources = []

    try:
        description = fetch_repo_metadata(repo)
        manifest_paths = fetch_manifest_paths(repo)
        readme = fetch_readme_content(repo)

        if any(k in path.lower() for k in keywords for path in manifest_paths):
            removal_sources.append("manifest_path")

        if any(k in repo.lower() for k in keywords):
            removal_sources.append("repo_name")

        if any(
            re.search(rf'(?<!\S){re.escape(k)}(?!\S)', str(description), re.IGNORECASE) and
            not any(re.search(rf'(?<!\S){re.escape(word)}\s+(\S+\s+){{0,4}}{re.escape(k)}(?!\S)', str(description), re.IGNORECASE)
                    for word in not_preceded_by)
            for k in keywords
        ):
            removal_sources.append("description")

        if any(
            re.search(rf'(?<!\S){re.escape(k)}(?!\S)', readme, re.IGNORECASE) and
            any(re.search(rf'(?<!\S){re.escape(p)}\s+(\S+\s+){{0,4}}{re.escape(k)}(?!\S)', readme, re.IGNORECASE)
                for p in preceded_by)
            for k in keywords
        ):
            removal_sources.append("readme")

        flag = "yes" if removal_sources else "no"
        df.at[idx, "removal_keyword_flag"] = flag
        df.at[idx, "Valid_Repo_Step3"] = "no" if flag == "yes" else "yes"
        df.at[idx, "removal_reason"] = ", ".join(removal_sources) if removal_sources else "none"

    except Exception as e:
        print(f"❌ Error processing {repo}: {e}")
        df.at[idx, "removal_keyword_flag"] = "error"
        df.at[idx, "Valid_Repo_Step3"] = "no"
        df.at[idx, "removal_reason"] = "error"
        sleep(1)

    
    df.to_csv(output_path, index=False)

# Remove temporary column before final save
if "Repository" in df.columns:
    df.drop(columns=["Repository"], inplace=True)

df.to_csv(output_path, index=False)
print(f"✅ Step 3 complete: {output_path} saved.")




[1/28250] Reviewing sintaxi/phonegap...
[2/28250] Reviewing rhomobile/rhodes...
[3/28250] Reviewing bradfitz/android-garage-opener...
[4/28250] Reviewing Dawnthorn/nagare...
[5/28250] Reviewing jamplus/jamplus...
[6/28250] Reviewing bpellin/keepassdroid...
[7/28250] Reviewing samuelclay/NewsBlur...
[8/28250] Reviewing connectbot/connectbot...
[9/28250] Reviewing JakeWharton/SMSMorse...
[10/28250] Reviewing JakeWharton/SMSBarrage...
[11/28250] Reviewing millenomi/diceshaker...
[12/28250] Reviewing simpligility/android-maven-plugin...
[13/28250] Reviewing pocmo/Yaaic...
[14/28250] Reviewing ushahidi/Ushahidi_Android...
[15/28250] Reviewing Ramblurr/Anki-Android...
[16/28250] Reviewing novoda/android-demos...
[17/28250] Reviewing commonsguy/cw-advandroid...
[18/28250] Reviewing commonsguy/cwac-merge...
[19/28250] Reviewing konklone/congress-android...
[20/28250] Reviewing johannilsson/sthlmtraveling...
[21/28250] Reviewing tidev/titanium-sdk...
[22/28250] Reviewing appcelerator-archive/sa